# Farm Disease-Risk Profile Discovery Through Unsupervised Pattern Clustering
**Author:** Antigravity Team  
**Dataset:** Kaggle "Animal Disease Prediction" (43,778 records, 7 attributes)  
**Methodology:** Unsupervised Pattern Discovery (Euclidean K-Means vs Manhattan K-Medians)  
**Objective:** Discover latent syndromic disease-risk profiles without diagnostic label leakage.

---

## Executive Summary & Workflow Overview
In livestock herd epidemiology, early risk stratification is critical for preventing catastrophic outbreaks. While supervised classification relies on confirmed post-mortem or laboratory diagnoses, field veterinarians often face unlabelled, syndromic presentations where latent risk clusters must be discovered organically.

This notebook executes a rigorous 3-objective data-science methodology:
1. **Objective 1: Data Preparation & Preprocessing:** Audits missing values, outliers, and duplicates; transforms arbitrary 3-slot categorical symptoms into a 24-dimensional multi-hot space; standardizes continuous vitals; isolates the ground-truth `Disease` label to ensure zero target leakage.
2. **Objective 2: Clustering Workflow:** Benchmarks standard Euclidean K-Means against a custom Manhattan K-Medians across $k \in [2, 10]$; evaluates WCSS/L1 cost, Silhouette Score, and Davies-Bouldin Index; conducts PCA scree and scatter analyses; defends selection of $k=4$.
3. **Objective 3: Interpretation & Responsible Reporting:** Profiles the discovered phenotypes with cautious naming; computes ANOVA feature contributions and surrogate tree fidelity; evaluates seed, bootstrap, and distance metric stability; documents clinical limitations and responsible use.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.stats import f_oneway
from sklearn.tree import DecisionTreeClassifier

# Configure aesthetic plotting
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams.update({"figure.dpi": 150, "font.size": 10})

print("Libraries imported successfully.")

---
## Objective 1: Data Preparation & Preprocessing

### 1.1 Dataset Ingestion & Initial Audit
We begin by inspecting the raw CSV file. The dataset contains 43,778 records with livestock species, vital signs (Temperature), demographics (Age), and three symptom slots, alongside the ground-truth diagnostic label (`Disease`).

In [ ]:
df_raw = pd.read_csv('../data/animal_disease_dataset.csv')
print(f"Dataset Shape: {df_raw.shape}")
print("\nData Types:\n", df_raw.dtypes)
print("\nFirst 5 Records:")
display(df_raw.head())

### 1.2 Variable Relevance Review
In accordance with unsupervised clustering principles:
- **`Animal` (Categorical):** Retained and one-hot encoded (4 species: cow, buffalo, goat, sheep).
- **`Age` & `Temperature` (Numeric):** Retained and standardized using Z-score scaling.
- **`Symptom 1`, `Symptom 2`, `Symptom 3` (Categorical):** Converted to a 24-dimensional binary multi-hot feature space.
- **`Disease` (Ground-Truth Diagnosis):** **CRITICAL STEP: DROPPED FROM CLUSTERING.** It is strictly isolated for post-hoc descriptive evaluation and never used for distance calculations.

In [ ]:
# Verify Missing Values
print("Missing Value Counts per Column:")
print(df_raw.isnull().sum())

# Outlier Audit via IQR
for col in ['Age', 'Temperature']:
    q25 = df_raw[col].quantile(0.25)
    q75 = df_raw[col].quantile(0.75)
    iqr = q75 - q25
    low = q25 - 1.5 * iqr
    high = q75 + 1.5 * iqr
    outliers = ((df_raw[col] < low) | (df_raw[col] > high)).sum()
    print(f"{col} - Q1: {q25:.2f}, Q3: {q75:.2f}, IQR: {iqr:.2f}, Bounds: [{low:.2f}, {high:.2f}] -> Outliers: {outliers}")

### 1.3 Feature Engineering: Multi-Hot Symptoms & One-Hot Species
Why is Label Encoding mathematically invalid?
> **Label Encoding Flaw:** Assigning ordinal integers (e.g. Buffalo=0, Cow=1, Goat=2, Sheep=3) creates artificial distance relationships: $(Sheep - Buffalo)^2 = 9$ vs $(Cow - Buffalo)^2 = 1$. Distance-based algorithms (K-Means) interpret these as spatial geometric distances. One-hot and multi-hot encoding ensure all categorical attributes are equidistant.

In [ ]:
# 1. Isolate Disease label
disease_labels = df_raw['Disease'].copy()
features_raw = df_raw.drop(columns=['Disease'])

# 2. One-hot encode Animal
animal_dummies = pd.get_dummies(features_raw['Animal'], prefix='animal', dtype=float)

# 3. Multi-hot encode Symptoms across all 3 slots
all_symptoms = sorted(list(set(features_raw['Symptom 1']).union(set(features_raw['Symptom 2'])).union(set(features_raw['Symptom 3']))))
symptom_mat = np.zeros((len(df_raw), len(all_symptoms)), dtype=float)

for i, s in enumerate(all_symptoms):
    mask = (features_raw['Symptom 1'] == s) | (features_raw['Symptom 2'] == s) | (features_raw['Symptom 3'] == s)
    symptom_mat[:, i] = mask.astype(float)
    
symptom_cols = [f"symptom_{s.replace(' ', '_')}" for s in all_symptoms]
symptom_df = pd.DataFrame(symptom_mat, columns=symptom_cols)

# 4. Standardize numeric features
scaler = StandardScaler()
scaled_num = pd.DataFrame(scaler.fit_transform(features_raw[['Age', 'Temperature']]), columns=['age_std', 'temp_std'])

# 5. Master Feature Matrix X
X = pd.concat([scaled_num, animal_dummies, symptom_df], axis=1)
print(f"Final Preprocessed Matrix X Shape: {X.shape}")

---
## Objective 2: Clustering Workflow (Euclidean K-Means vs Manhattan K-Medians)

### 2.1 Algorithm Implementation & Distance Metrics
Standard K-Means minimizes squared Euclidean (L2) errors using the cluster mean. For sparse binary symptom indicator spaces, Manhattan (L1) distance directly counts mismatched binary flags (Hamming distance). The centroid that minimizes Manhattan distance is the coordinate-wise median (K-Medians).

In [ ]:
# Load precomputed metrics from pipeline to avoid redundant re-computation
with open('../outputs/metrics.json') as f:
    metrics_data = json.load(f)

df_metrics = pd.DataFrame(metrics_data['records'])
print("Clustering Metrics Summary across k=2..10:")
display(df_metrics.head(18))

### 2.2 Elbow, Silhouette, and Davies-Bouldin Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Elbow (WCSS / Cost)
df_km = df_metrics[df_metrics['algorithm'] == 'K-Means']
df_kmed = df_metrics[df_metrics['algorithm'] == 'K-Medians']

axes[0].plot(df_km['k'], df_km['cost_wcss'], marker='o', label='Euclidean K-Means', color='#1976d2')
axes[0].axvline(x=4, color='#d32f2f', linestyle='--', label='k=4 Selected')
axes[0].set_title("Elbow Curve: WCSS vs k")
axes[0].set_xlabel("Clusters (k)")
axes[0].set_ylabel("Inertia / WCSS")
axes[0].legend()

# Silhouette
axes[1].plot(df_km['k'], df_km['silhouette'], marker='o', label='Euclidean K-Means', color='#1976d2')
axes[1].plot(df_kmed['k'], df_kmed['silhouette'], marker='s', label='Manhattan K-Medians', color='#388e3c')
axes[1].set_title("Silhouette Score vs k (Higher is Better)")
axes[1].set_xlabel("Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].legend()

# Davies-Bouldin
axes[2].plot(df_km['k'], df_km['davies_bouldin'], marker='o', label='Euclidean K-Means', color='#1976d2')
axes[2].plot(df_kmed['k'], df_kmed['davies_bouldin'], marker='s', label='Manhattan K-Medians', color='#388e3c')
axes[2].axvline(x=4, color='#d32f2f', linestyle='--', label='k=4 Optimal DBI (1.9161)')
axes[2].set_title("Davies-Bouldin Index vs k (Lower is Better)")
axes[2].set_xlabel("Clusters (k)")
axes[2].set_ylabel("DBI")
axes[2].legend()

plt.tight_layout()
plt.show()

### 2.3 PCA Projection & Variance Analysis

In [ ]:
pca = PCA(n_components=3, random_state=42)
coords_3d = pca.fit_transform(X)

print("Explained Variance Ratios (Top 3 Components):")
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1}: {var*100:.2f}% (Cumulative: {np.sum(pca.explained_variance_ratio_[:i+1])*100:.2f}%)")

### 2.4 Configuration Selection Justification
We formally select **$k=4$ with Euclidean K-Means**:
1. **Global Minimum Davies-Bouldin Index (1.9161):** Represents the sharpest boundary separation relative to cluster compactness.
2. **Robust Silhouette Score (0.1405):** Significantly superior to $k=5..10$.
3. **WCSS Elbow Inflection:** Beyond $k=4$, marginal inertia reductions plateau.
4. **Clinical Granularity:** Unlike $k=2$ which simply splits age/temperature, $k=4$ resolves four distinct syndromic complexes.

---
## Objective 3: Interpretation & Responsible Reporting

### 3.1 Cluster Profiles & Clinical Interpretation
Let us inspect the four discovered risk profiles.

In [ ]:
with open('../outputs/cluster_profiles.json') as f:
    profiles_data = json.load(f)

for cid, prof in profiles_data['profiles'].items():
    print(f"\n{'='*60}")
    print(f"{prof['display_name']} ({prof['sample_count']} animals, {prof['percentage']}%)")
    print(f"Summary: {prof['summary']}")
    print(f"Age Mean: {prof['age_stats']['mean']} yrs | Temp Mean: {prof['temp_stats']['mean']} F")
    print("Species Breakdown:", prof['species_distribution'])
    print("Top Symptoms:")
    for s in prof['top_symptoms'][:4]:
        print(f"  - {s['symptom']}: {s['prevalence_pct']}%")

### 3.2 Key Feature Contributions (ANOVA F-Statistic)

In [ ]:
df_anova = pd.DataFrame(profiles_data['anova_ranking']).head(10)
print("Top 10 Distinguishing Features by ANOVA F-Statistic:")
display(df_anova[['clean_name', 'f_statistic', 'p_value']])

### 3.3 Post-Hoc Descriptive Cross-Check with Disease Labels
> **DISCLAIMER:** Ground-truth diagnoses were completely excluded during clustering. This cross-check is purely an observational audit to see how phenotypic symptom clusters map to clinical conditions.

In [ ]:
crosscheck = profiles_data['disease_crosscheck']
ct_data = crosscheck['contingency_matrix']
diseases = crosscheck['diseases']

matrix = [[ct_data[c][d]['count'] for d in diseases] for c in sorted(ct_data.keys(), key=int)]
df_cross = pd.DataFrame(matrix, index=[f"Profile {int(c)+1}" for c in sorted(ct_data.keys(), key=int)], columns=diseases)

plt.figure(figsize=(10, 5))
sns.heatmap(df_cross, annot=True, fmt='d', cmap='YlGnBu')
plt.title("Contingency Audit: Unsupervised Profiles vs Known Diagnoses")
plt.xlabel("Disease (Ground Truth)")
plt.ylabel("Discovered Cluster")
plt.show()

### 3.4 Stability Analysis
We evaluate Adjusted Rand Index (ARI) across random initialization seeds, bootstrap subsamples, and distance metrics.

In [ ]:
with open('../outputs/stability.json') as f:
    stability_data = json.load(f)

print(f"Mean Seed ARI (5 restarts): {stability_data['seed_stability']['mean_ari']:.4f}")
print(f"Mean Bootstrap Overlap ARI (80% sampling): {stability_data['bootstrap_stability']['mean_overlap_ari']:.4f}")
print(f"\nVerdict:\n{stability_data['verdict']}")

### 3.5 Limitations & Responsible Use
- **Exploratory, Not Diagnostic:** Discovered clusters represent observational patterns, never definitive pathological diagnoses.
- **Veterinary Supervision:** Must not be used for automated treatment, antibiotic administration, or culling.
- **Metric Considerations:** While continuous vitals suit Euclidean K-Means, binary symptoms suit Manhattan/Hamming distance; both metrics should be cross-examined in real herd workflows.